# Loading Packages and data

dicom_parser documentation: https://dicom-parser.readthedocs.io/en/latest/quickstart.html

In [1]:
## pip install dicom-parser Pillow python-gdcm pylibjpeg
from dicom_parser import Image
from dicom_parser import Series

image = Image('./data/image-00178.dcm')
series = Series('./data/example_files_dcm/series-00000_from_So25/')

# Parsing a DICOM header

DICOM reference: https://www.dicomlibrary.com/dicom/dicom-tags/

dicom_parser value representation: https://dicom-parser.readthedocs.io/en/latest/value_representation.html

In [2]:
image.header

/Users/jingwenluo/Desktop/FU/Computer vision for biomedical images S26 Resources/.venv/lib/python3.9/site-packages/pydicom/valuerep.py:290: UserWarning: Invalid value for VR UI: '1.2.826.0.1.3680043.8.1055.1.20111102150758591.96842950.07877442'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warnings.warn(msg)


(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL', 'HELIX']
(0008, 0012) Instance Creation Date              DA: '20061012'
(0008, 0013) Instance Creation Time              TM: '091652.000000'
(0008, 0016) SOP Class UID                       UI: CT Image Storage
(0008, 0018) SOP Instance UID                    UI: 1.2.826.0.1.3680043.8.1055.1.20111102150831341.79554570.78674210
(0008, 0020) Study Date                          DA: '20061012'
(0008, 0022) Acquisition Date                    DA: '20061012'
(0008, 0023) Content Date                        DA: '20061012'
(0008, 0030) Study Time                          TM: '090258.000000'
(0008, 0032) Acquisition Time                    TM: '085229.000000'
(0008, 0033) Content Time                        TM: '085235.280998'
(0008, 0060) Modality                            CS: 'CT'
(0008, 1030) Study Description                   LO: 'CT1 abd

In [3]:
raw_value_by_tag = image.header.raw[(0x0018, 0x0050)].value
print("Raw Value(Tag):", raw_value_by_tag, "\t", "Type:", type(raw_value_by_tag))

raw_value_by_name = image.header.raw['SliceThickness'].value
print("Raw Value(Name):", raw_value_by_name, "\t",  "Type:", type(raw_value_by_name))


# pasrsing values using get method return different types of value compared to raw value
# The type of parsed value depends on the tag and the content of the tag. refer to the dicom_parser documentation for details.
parsed_value_wrong = image.header.get('ImagingFrequency')
print("Parsed Value(not matching):", parsed_value_wrong, "\t", "Type:", type(parsed_value_wrong))

parsed_value = image.header.get('SliceThickness')
print("Parsed Value:", parsed_value, "\t", "Type:", type(parsed_value))

Raw Value(Tag): 1.0 	 Type: <class 'pydicom.valuerep.DSfloat'>
Raw Value(Name): 1.0 	 Type: <class 'pydicom.valuerep.DSfloat'>
Parsed Value(not matching): None 	 Type: <class 'NoneType'>
Parsed Value: 1.0 	 Type: <class 'float'>


# Exercise: DICOM tags

Useful resources: https://dicom.innolitics.com/ciods, http://dicomlookup.com/(✨)

1. What imaging technique was used on the patient? On which body part?

In [4]:
Modality = image.header.get('Modality')
print("Modality:", Modality, "\t", "Type:", type(Modality))

body_part_examined = image.header.get('BodyPartExamined')
print("Body Part Examined:", body_part_examined, "\t", "Type:", type(body_part_examined))

region = image.header.get('AcquisitionRegionSequence')
print("Region:", region, "\t", "Type:", type(region))

study_description = image.header.get('StudyDescription')
print("Study Description:", study_description, "\t", "Type:", type(study_description))

Modality: Computed Tomography 	 Type: <class 'str'>
Body Part Examined: None 	 Type: <class 'NoneType'>
Region: None 	 Type: <class 'NoneType'>
Study Description: CT1 abdomen 	 Type: <class 'str'>


##### Question: why stored in 'StudyDescription' instead of 'BodyPartExamined'

2. When was the image taken?

In [5]:
StudyDate = image.header.get('StudyDate')
print("Study Date:", StudyDate, "\t", "Type:", type(StudyDate))

AcquisitionDate = image.header.get('AcquisitionDate')
print("Acquisition Date:", AcquisitionDate, "\t", "Type:", type(AcquisitionDate))

ContentDate= image.header.get('ContentDate')
print("Content Date:", ContentDate, "\t", "Type:", type(ContentDate))

StudyTime = image.header.get('StudyTime')
print("Study Time:", StudyTime, "\t", "Type:", type(StudyTime))

AcquisitionTime = image.header.get('AcquisitionTime')
print("Acquisition Time:", AcquisitionTime, "\t", "Type:", type(AcquisitionTime))

ContentTime = image.header.get('ContentTime')
print("Content Time:", ContentTime, "\t", "Type:", type(ContentTime))

Study Date: 2006-10-12 	 Type: <class 'datetime.date'>
Acquisition Date: 2006-10-12 	 Type: <class 'datetime.date'>
Content Date: 2006-10-12 	 Type: <class 'datetime.date'>
Study Time: 09:02:58 	 Type: <class 'datetime.time'>
Acquisition Time: 08:52:29 	 Type: <class 'datetime.time'>
Content Time: 08:52:35.280998 	 Type: <class 'datetime.time'>


3. What is the patient position in the CT machine? What would the code for the opposite orientation be?
   
   - opposite_mapping refer to: https://dicom.innolitics.com/ciods/intraocular-lens-calculations/general-series/00185100

In [6]:
patientPosition_raw = image.header.raw['PatientPosition'].value
print("Patient Position Raw:", patientPosition_raw, "\t", "Type:", type(patientPosition_raw))
patientPosition = image.header.get('PatientPosition')
print("Patient Position:", patientPosition, "\t", "Type:", type(patientPosition))

opposite_map = {
    'HFS': 'HFP',
    'HFP': 'HFS',
    'FFS': 'FFP',
    'FFP': 'FFS'
}
print("Opposite:", opposite_map.get(patientPosition_raw, 'Unknown')) # if the key does not exist, return 'unknown' instead of None
print("Opposite:", opposite_map.get(patientPosition, 'Unknown')) # if the key does not exist, return 'unknown' instead of None

Patient Position Raw: FFS 	 Type: <class 'str'>
Patient Position: Feet First-Supine 	 Type: <class 'str'>
Opposite: FFP
Opposite: Unknown


##### Question: 
- What is the correct way to define a mapping to the opposite patient orientation?

- Should this mapping handle both head/foot reversal and supine/prone reversal?

- In medical image processing, why is it important to consider the opposite of PatientPosition, how does it affect image alignment, standardization, and comparison?

4. What is the xyz image size in millimeters?

In [7]:
# PixelSpacing is the size of one pixel in mm, usually [row_spacing, column_spacing].
# Rows is the number of pixels vertically.
# Columns is the number of pixels horizontally.

pixel_spacing = image.header.get('PixelSpacing')  # [row, column] in mm
rows = image.header.get('Rows')
cols = image.header.get('Columns')

slice_thickness = image.header.get('SliceThickness')
# SpacingBetweenSlices is the distance between the centers of adjacent slices in mm. 
slice_spacing = image.header.get('SpacingBetweenSlices') 
num_slices = len(series.images)

# there are cols pixels horizontally and each pixel width is pixel_spacing[1]
x_mm = cols * pixel_spacing[1] 
# there are rows pixels vertically and each pixel height is pixel_spacing[0]
y_mm = rows * pixel_spacing[0] 
# num_slices slices have num_slices - 1 gaps between their centers
z_mm = (num_slices-1) * slice_spacing 
print("Image size X (mm):", cols ,"*", pixel_spacing[1], "=", x_mm)
print("Image size Y (mm):", rows ,"*", pixel_spacing[0], "=", y_mm)
print("Image size Z (mm):", "(",num_slices ,"-1) *", slice_spacing, "=", z_mm)


Image size X (mm): 512 * 0.58984375 = 302.0
Image size Y (mm): 512 * 0.58984375 = 302.0
Image size Z (mm): ( 361 -1) * 0.5 = 180.0


##### Question: How to get the number of slice without having series files?

5. What image compression format is used for the pixel data? What is the bit depth?

In [8]:
imageType = image.header.get('ImageType')
print("Image Type:", imageType, "\t", "Type:", type(imageType))
lossyCompression = image.header.get('LossyImageCompression')
print("Lossy Image Compression:", lossyCompression)   
lossyCompressionRatio = image.header.get('LossyImageCompressionRatio')
print("Lossy Image Compression Ratio:", lossyCompressionRatio)  
lossy_method = image.header.get('LossyImageCompressionMethod') 
print("Lossy Image Compression Method:", lossy_method)

# Number of bits allocated for each pixel sample. 
# Each sample shall have the same number of bits allocated. 
# Bits Allocated (0028,0100) shall be either 1, or a multiple of 8.
print("Bits Allocated:", image.header.get("BitsAllocated"))


# Number of bits stored for each pixel sample. Each sample shall have the same number of bits stored.
# Example Values: 16, 12, 8, etc. The value of BitsStored shall be less than or equal to the value of BitsAllocated.
print("Bits Stored:", image.header.get("BitsStored"))

# Most significant bit for pixel sample data. 
# Each sample shall have the same high bit. 
# High Bit (0028,0102) shall be one less than Bits Stored (0028,0101).
print("high_bit:", image.header.get("HighBit"))

Image Type: ('ORIGINAL', 'PRIMARY', 'AXIAL', 'HELIX') 	 Type: <class 'tuple'>
Lossy Image Compression: 01
Lossy Image Compression Ratio: 6.019104
Lossy Image Compression Method: None
Bits Allocated: 16
Bits Stored: 12
high_bit: 11


6. What is the tag for the patient name? Do we have that information?

In [9]:
patientName_raw = image.header.raw['PatientName']
print("Patient Name (Raw):", patientName_raw, "\t", "Type:", type(patientName_raw))

patientName = image.header.get('PatientName')
print("Patient Name:", patientName, "\t", "Type:", type(patientName))

Patient Name (Raw): (0010, 0010) Patient's Name                      PN: 'Anonymized' 	 Type: <class 'pydicom.dataelem.DataElement'>
Patient Name: {'name_prefix': '', 'given_name': '', 'middle_name': '', 'family_name': 'Anonymized', 'name_suffix': ''} 	 Type: <class 'dict'>


# Loading NIFTI files


In [10]:
import  nibabel

In [11]:
img_nii = nibabel.load('./data/example_files_nii/ID_0011fe81e.nii')

In [12]:
print(img_nii.header)

<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : b''
db_name         : b''
extents         : 0
session_error   : 1
regular         : b'r'
dim_info        : 0
dim             : [   3 1024 1024    1    1    1    1    1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : uint8
bitpix          : 8
slice_start     : 0
pixdim          : [1.    0.139 0.139 1.    0.    0.    0.    0.   ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 0
glmin           : 0
descrip         : b'Time=0.000'
aux_file        : b''
qform_code      : unknown
sform_code      : unknown
quatern_b       : 0.0
quatern_c       : 0.0
quatern_d       : 0.0
qoffset_x       : 0.0
qoffset_y       : -1023.0
qoffset_z       : 0.0
srow_x    

# Exercise: NIFTI images

Load the paired .nii and .dcm file

Examine the headers. What differences do you notice?

In [13]:
img_dcm = Image('./data/example_files_nii/ID_0011fe81e.dcm')